# BE Group - marketing channel economics

### Read this top to bottom and you have the whole project

Every table, every check and every finding is in here, with the code that
produced it. Nothing requires opening another file.

**What this is.** Three operating workbooks from a Vietnamese mobility and
delivery platform, handed over as a business-analysis exercise. They were
originally answered in Excel. This is the rebuild: parsed reproducibly,
analysed further, and checked before anything was quoted.


> ## Every number below is indexed. None of them is real.
>
> The source is BE Group's own operating data. Nothing real is published in
> this repository and nothing real can be recovered from what is.
>
> - **Levels** - GMV, cost, users, orders, commission - are rescaled so the
>   first period of each series equals 100.
> - **Ratios** - CAC, CIR, take rate, margin, conversion, retention - are
>   multiplied by one undisclosed constant per metric family.
>
> **Hidden:** every currency amount, every count, and every rate as a level -
> GMV, spend, budget, commission, users, installs, CAC, CIR, take rate,
> margin rate, retention.
>
> **Still real, deliberately:** shares and mixes, ratios between two indexed
> figures, rankings, counts of things rather than of money (54 weeks checked,
> 7 brands negative, 8 of 8 days), and rates of change in percentage points.
> Without those there is no analysis left to read.
>
> So you can learn here that one channel delivers two thirds of acquisition at a
> fraction of the cost of another. You cannot learn what either cost.
> Part 11 shows the mechanism and proves the findings survive it.


### The route through

| Part | | 
|---|---|
| 1 | What the source actually looks like, and why nothing can read it |
| 2 | The parser, and the four traps in it |
| 3 | What came out |
| 4 | Channel P&L |
| 5 | Check: does the channel split sum to the company total? |
| 6 | Check: does reported CAC match its own formula? |
| 7 | Check: do the two cities sum to "All Viet Nam"? **This one found something** |
| 8 | Brand economics, and the take rate the source cannot show |
| 9 | The promotion case: what a commission increase bought, and how fast it went |
| 10 | Payback, and every assumption behind it |
| 11 | The indexing layer |
| 12 | What this data cannot answer |


In [1]:
import json
import sys
from pathlib import Path

import duckdb
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PUBLISHED = ROOT / "data" / "published"
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 46)
# Previews show at least 10 rows throughout. Three rows is not enough to see a
# pattern, a null cluster, or a grain finer than you assumed.
pd.set_option("display.max_rows", 40)
pd.set_option("display.min_rows", 20)

db = duckdb.connect(":memory:")
for parquet in sorted(PUBLISHED.glob("*.parquet")):
    db.execute(
        f"CREATE OR REPLACE VIEW {parquet.stem} AS "
        f"SELECT * FROM read_parquet('{parquet.as_posix()}')"
    )


def q(sql):
    """Run a query and return a DataFrame."""
    return db.execute(sql).fetchdf()


def show_py(filename, func, lines=None):
    """Print the production function that builds something, read at run time.

    Nothing here is a copy of the code. It is the code, read out of src/ as the
    cell runs, so it cannot fall out of step with what actually executed.
    """
    path = ROOT / "src" / filename
    rows = path.read_text(encoding="utf-8").splitlines()
    start = next(
        (i for i, row in enumerate(rows)
         if row.startswith((f"def {func}(", f"{func} = ", f"{func} = {{", f"{func}: "))),
        None,
    )
    if start is None:
        print(f"{func} not found in {filename}")
        return
    end = start + 1
    while end < len(rows) and not rows[end].startswith(("def ", "class ", "@")):
        end += 1
    body = rows[start:end]
    if lines:
        body = body[:lines]
    print(f"# {filename}, line {start + 1}\n")
    print("\n".join(body).rstrip())


print("views:", ", ".join(sorted(p.stem for p in PUBLISHED.glob("*.parquet"))))


views: mart_brand_month_economics, mart_channel_payback, mart_channel_week_pnl, mart_promo_unit_economics, qa_channel_reconciliation, qa_city_reconciliation, qa_completeness


---

## Part 1 - What the source actually looks like

**The problem.** These are not datasets. They are reporting outputs - the
weekly deck an operating team actually keeps - and nothing downstream can
read them.

| | |
|---|---|
| **Built by** | nothing yet - this is the input |
| **Written to** | `data/source/*.xlsx` (git-ignored, never published) |


The channel workbook is **186 rows by 56 columns**, and laid out like this:

```
       col 0                       col 1        col 2        col 3    ...
row 0  weekly                                   Mon          Mon
row 1                                           2024-12-23   2024-12-30
row 2  TOP LINES                      <- a banner, not data
row 3  Open App User (Lag T-2)                   2060798      2179423
row 4  Installs (non PA)                          1413086      1079737
...
row 40                                 <- a blank spacer row
row 41 BREAK BY CHANNELS               <- another banner
row 42 Install (UA)                AllBe         ...
row 43 Install (UA)                Affiliate     ...   <- same metric,
row 44 Install (UA)                Google Ads    ...      second key in col 1
...
row 141 CAC (=Budget RFT_raw / RFT)  AllBe        ...   <- formula in the label
row 142 CAC                          Affiliate    ...   <- bare name below it
```

Six separate problems in one sheet:

| | |
|---|---|
| Metrics run **down** the rows, dates run **across** the columns | the opposite of what any join needs |
| A second grouping key hides in **column B** | the channel, present only below row 41 |
| **Section banners** sit inside the data | `TOP LINES`, `BREAK BY CHANNELS` |
| **Blank spacer rows** separate blocks | row 40, row 50, row 59... |
| The **first row of each block** carries the formula in its label | `CAC (=Budget RFT_raw / RFT)` vs `CAC` |
| **Reporting lags** are written into the metric name | `(Lag T-2)`, `(lag T-1)` |


> **Conclusion.** This is what a real handover looks like. A public dataset arrives tidy; an operating team's workbook never does. Turning it into something that can be joined is the least glamorous part of the project and the part everything else depends on.


---

## Part 2 - The parser, and the four traps in it

**What it does.** Reads the workbooks exactly as delivered and emits long-format
tables with explicit keys. Four things it has to get right that a naive read
misses - each one silently corrupts a downstream number.

| | |
|---|---|
| **Built by** | `src/parse_source.py` |
| **Written to** | `data/private/*.parquet` (git-ignored - these hold real values) |


| # | Trap | What happens if you miss it |
|---|---|---|
| 1 | **Inline subtotals** - `beBike Total` sits in the same column as `beBike` | every aggregate double-counts. **80 rows of 544** |
| 2 | **Two dimensions stacked** - service lines and rider personas share one block | you average a persona against a product line |
| 3 | **Metric labels that differ by row** - the formula on row 1 of each block, the bare name on rows 2-8 | `GROUP BY metric` splits one series into two |
| 4 | **Reporting lags in the label** - `(Lag T-2)` on installs, `(lag T-1)` on new installs | any week-on-week ratio across differently-lagged metrics is wrong |


### Trap 1 and 2, handled - this is the production code, read at run time


In [2]:
show_py("parse_source.py", "parse_promo_usage", lines=46)

# parse_source.py, line 268

def parse_promo_usage(path: Path) -> pd.DataFrame:
    """Promo per trip at M0 and M1, two side-by-side blocks.

    Two things in this sheet will quietly wreck an aggregate if they are not
    handled here.

    First, the block stacks TWO dimensions under one header: service types
    (beBike, beCar, beClean, beDelivery, beFood) and rider personas (Office
    Worker, University Student, Parent, ...). They are not comparable and must
    not sit in one column, so they are labelled and split.

    Second, it contains subtotal rows - "beBike Total", "beFood Total" - inline
    with the detail. Summing the column as delivered double-counts every one of
    them. They are dropped, and the drop is counted so it is visible.
    """
    raw = pd.read_excel(path, sheet_name="Data 2", header=None)
    rows = []
    # The M0 block starts at column 1, the M1 block at column 10. Row 6 holds
    # the month headers for both.
    for cohort, key_column in (("M0", 1), ("

### Trap 3 and 4

The lag is kept as a **column** rather than stripped, because it is part of the
metric's definition. So are the accounting qualifiers - `n/VAT`, `w/Fee`,
`n/CIT` - which move to a `basis` column instead of being deleted.

A figure `w/Fee` and a figure `n/Fee` are different numbers for the same thing.
Delete the suffix and they look identical.


In [3]:
show_py("parse_source.py", "split_basis")

# parse_source.py, line 91

def split_basis(label: str) -> tuple[str, str]:
    """Separate 'Digital Cost w/Fee n/CIT n/VAT' into the measure and its basis."""
    basis = ""
    for marker, meaning in BASIS_MARKERS:
        if marker in label:
            label = label.replace(marker, "")
            basis = meaning
            break
    return label.replace(",", " ").replace("_raw", "").strip(" ,"), basis


> **Conclusion.** None of these four is visible unless you look. All four change a number. That is the argument for parsing a workbook in code rather than by hand: the decisions get written down, and they get made the same way every time.


---

## Part 3 - What came out

**Eight tidy tables.** One row per observation, explicit keys, nothing merged.

| | |
|---|---|
| **Built by** | `src/parse_source.py` |
| **Written to** | `data/private/` then, indexed, `data/published/` |


In [4]:
print("Published tables (the indexed versions this notebook reads):\n")
for parquet in sorted(PUBLISHED.glob("*.parquet")):
    frame = pd.read_parquet(parquet)
    print(f"  {parquet.stem:<34} {len(frame):>6,} rows   {frame.shape[1]:>2} cols")

Published tables (the indexed versions this notebook reads):

  mart_brand_month_economics            679 rows   21 cols
  mart_channel_payback                    6 rows   11 cols
  mart_channel_week_pnl                 486 rows   24 cols
  mart_promo_unit_economics              12 rows   20 cols
  qa_channel_reconciliation              54 rows    7 cols
  qa_city_reconciliation                  5 rows    4 cols
  qa_completeness                         3 rows    4 cols


### What one of them looks like


In [5]:
q("SELECT * FROM mart_channel_week_pnl ORDER BY channel, week_start LIMIT 12")

,channel,week_start,channel_role,installs_index,signups_index,first_trip_users_index,requests_index,completed_bookings_index,gmv_index,aov_index,digital_cost_index,budget_first_trip_index,budget_trip_index,cac_reported_index,cpa_reported_index,cir_reported_index,completion_rate_index,cac_recomputed_index,cpa_recomputed_index,cir_recomputed_index,install_to_signup_index,signup_to_first_trip_index,request_to_booking_index,gmv_per_first_trip_user_index
0,Affiliate,2024-12-23,paid channel,356.81,233.22,255.54,341.02,321.01,326.76,107.23,213.43,223.30,82.40,392.362,0.0,1.490,167.408,391.914,0.003,1.484,5.940,180.342,166.995,196.10
1,Affiliate,2024-12-30,paid channel,270.45,281.59,289.03,358.95,320.83,341.60,111.89,225.58,236.00,91.39,366.773,0.0,1.490,158.479,366.213,0.004,1.500,9.462,168.934,158.568,181.26
2,Affiliate,2025-01-06,paid channel,190.88,285.83,341.43,393.27,345.06,361.30,110.02,253.33,265.03,101.87,347.581,0.0,1.583,156.247,348.133,0.004,1.593,13.608,196.605,155.661,162.29
3,Affiliate,2025-01-13,paid channel,225.33,360.25,432.27,448.47,365.97,398.47,114.69,311.61,326.05,107.87,339.052,0.0,1.769,145.086,338.287,0.004,1.777,14.529,197.487,144.771,141.37
4,Affiliate,2025-01-20,paid channel,184.70,324.09,351.53,401.45,305.91,364.74,125.41,253.82,265.58,91.39,339.052,0.0,1.583,136.158,338.830,0.004,1.581,15.946,178.523,135.187,159.12
5,Affiliate,2025-01-27,paid channel,163.52,221.99,141.51,158.99,87.27,146.95,177.16,105.31,110.22,25.47,349.714,0.0,1.630,98.212,349.328,0.004,1.628,12.337,104.922,97.386,159.25
6,Affiliate,2025-02-03,paid channel,145.16,245.20,225.61,296.71,233.99,285.74,128.67,162.85,170.36,67.42,339.052,0.0,1.304,140.622,338.660,0.004,1.295,15.350,151.442,139.905,194.23
7,Affiliate,2025-02-10,paid channel,178.37,233.00,218.65,315.70,267.78,297.72,117.02,164.25,171.80,76.40,351.846,0.0,1.257,149.551,352.405,0.004,1.253,11.871,154.446,150.484,208.83
8,Affiliate,2025-02-17,paid channel,204.31,218.39,202.71,300.52,268.04,285.32,111.89,157.94,165.20,76.40,364.640,0.0,1.257,158.479,365.498,0.004,1.258,9.714,152.772,158.237,215.86
9,Affiliate,2025-02-24,paid channel,154.18,197.36,182.65,305.97,277.25,286.77,108.62,146.72,153.48,62.92,377.435,0.0,1.164,160.711,376.874,0.003,1.162,11.632,152.319,160.760,240.80


> **Conclusion.** Every metric that was a block of rows is now a column on a record. Cost sits next to what the cost bought, which is the whole point of a channel P&L and is impossible in the source layout.


---

## Part 4 - Channel P&L

**The question.** What does each acquisition channel cost, and what does it return?

Nine channels appear in the source, but only seven are channels. `AllBe` is the
company total and `CPC (excl. AFF)` is a grouping of the paid ones. Including
either with the detail double-counts, so each row carries a `channel_role`.

| | |
|---|---|
| **Built by** | `src/build_marts.py` - `build_channel_pnl()` |
| **Written to** | `mart.channel_week_pnl` - 486 rows |


In [6]:
show_py("build_marts.py", "build_channel_pnl", lines=30)

# build_marts.py, line 56

def build_channel_pnl(connection) -> None:
    """One row per channel per week, with the economics on the same row.

    The source stores every metric as its own block of rows, so the first job
    is to pivot it back into a record. Only then can cost sit next to what the
    cost bought, which is the whole point of a channel P&L.
    """
    connection.execute(
        f"""
        CREATE OR REPLACE TABLE mart.channel_week_pnl AS
        WITH wide AS (
            SELECT
                channel,
                week_start,
                MAX(CASE WHEN metric = 'Install (UA)'       THEN value END) AS installs,
                MAX(CASE WHEN metric = 'SU'                 THEN value END) AS signups,
                MAX(CASE WHEN metric = 'RFT'                THEN value END) AS first_trip_users,
                MAX(CASE WHEN metric = 'Requests'           THEN value END) AS requests,
                MAX(CASE WHEN metric = 'Booking_Completed'  THEN value END) AS 

### Each channel over the whole 54 weeks


In [7]:
q("""
    SELECT channel, channel_role,
           COUNT(*)                                   AS weeks,
           ROUND(SUM(first_trip_users_index))         AS first_trip_users_idx,
           ROUND(SUM(budget_first_trip_index))        AS budget_idx,
           ROUND(AVG(cac_reported_index), 1)          AS cac_idx,
           ROUND(AVG(install_to_signup_index), 1)     AS install_to_signup_idx,
           ROUND(AVG(signup_to_first_trip_index), 1)  AS signup_to_rft_idx
    FROM mart_channel_week_pnl
    GROUP BY channel, channel_role
    ORDER BY first_trip_users_idx DESC
""")

,channel,channel_role,weeks,first_trip_users_idx,budget_idx,cac_idx,install_to_signup_idx,signup_to_rft_idx
0,AllBe,company total,54,24010.0,23632.0,450.6,15.6,157.3
1,Affiliate,paid channel,54,13113.0,9871.0,338.3,9.3,170.8
2,Organic,unpaid,54,4332.0,0.0,0.0,47.0,181.9
3,Apple Search Ads,paid channel,54,2245.0,4305.0,978.1,98.2,112.2
4,Facebook Ads,paid channel,54,2191.0,5168.0,1049.4,57.9,402.5
5,Google Ads,paid channel,54,1819.0,3991.0,995.4,82.8,78.9
6,Tiktok,paid channel,54,166.0,298.0,1683.7,202.7,481.8
7,Others,paid channel,54,145.0,0.0,0.0,116.3,150.3
8,CPC (excl. AFF),grouping,54,NaN,NaN,992.6,NaN,NaN


### Share of acquisition against share of budget

Shares are ratios, so the indexing does not touch them. These are the real
percentages.


In [8]:
q("""
    WITH paid AS (
        SELECT channel,
               SUM(first_trip_users_index)  AS rft,
               SUM(budget_first_trip_index) AS budget
        FROM mart_channel_week_pnl
        WHERE channel_role = 'paid channel'
        GROUP BY channel
    )
    SELECT channel,
           ROUND(100.0 * rft    / SUM(rft)    OVER (), 1) AS pct_of_first_trips,
           ROUND(100.0 * budget / SUM(budget) OVER (), 1) AS pct_of_budget,
           ROUND(100.0 * rft / SUM(rft) OVER ()
               - 100.0 * budget / SUM(budget) OVER (), 1) AS efficiency_gap_pts
    FROM paid ORDER BY pct_of_first_trips DESC
""")

,channel,pct_of_first_trips,pct_of_budget,efficiency_gap_pts
0,Affiliate,66.6,41.8,24.9
1,Apple Search Ads,11.4,18.2,-6.8
2,Facebook Ads,11.1,21.9,-10.7
3,Google Ads,9.2,16.9,-7.6
4,Tiktok,0.8,1.3,-0.4
5,Others,0.7,0.0,0.7


> **Conclusion.** Affiliate delivers 66.6% of paid first-trip users on 41.8% of the budget - 25 points of positive gap. Facebook Ads and Google Ads together are 20.4% of first trips on 38.8% of budget. Read at face value the budget decision looks obvious. Part 12 is about why it should not be read at face value.


---

## Part 5 - Check: does the channel split sum to the company total?

**Why it matters.** Every share in Part 4 assumes the per-channel rows add up to
the company row. If they do not, every percentage above is computed on a base
that does not exist.

This is the first thing to test on any table that carries both a total and its
components, and the source does not test it.

| | |
|---|---|
| **Built by** | `src/build_marts.py` - `build_qa()` |
| **Written to** | `qa.channel_reconciliation` - one row per week |


In [9]:
q("""
    SELECT week_start,
           ROUND(total_rft_index, 2)   AS company_total_idx,
           ROUND(detail_rft_index, 2)  AS sum_of_channels_idx,
           ROUND(rft_delta_pct * 100, 6) AS delta_pct,
           ROUND(gmv_delta_pct * 100, 6) AS gmv_delta_pct
    FROM qa_channel_reconciliation
    ORDER BY ABS(rft_delta_pct) DESC LIMIT 12
""")

,week_start,company_total_idx,sum_of_channels_idx,delta_pct,gmv_delta_pct
0,2025-12-22,126.45,126.45,-0.003728,0.000000
1,2025-02-24,79.49,79.49,-0.002965,0.000011
2,2025-03-24,81.61,81.62,0.002888,0.000012
3,2025-07-21,84.00,84.01,0.002806,0.000012
4,2025-03-03,86.46,86.46,-0.002726,-0.000011
5,2025-03-31,89.13,89.13,0.002645,0.000000
6,2025-02-17,90.75,90.75,-0.002598,0.000011
7,2025-07-28,93.55,93.55,-0.002520,0.000011
8,2025-05-05,93.78,93.78,0.002514,-0.000010
9,2025-06-02,93.94,93.93,-0.002510,0.000011


In [10]:
q("""
    SELECT COUNT(*)                                        AS weeks_checked,
           ROUND(MAX(ABS(rft_delta_pct)) * 100, 6)         AS worst_delta_pct,
           COUNT(*) FILTER (WHERE ABS(rft_delta_pct) < 0.0001) AS within_0_01_pct,
           ROUND(MAX(ABS(gmv_delta_pct)) * 100, 6)         AS worst_gmv_delta_pct
    FROM qa_channel_reconciliation
""")

,weeks_checked,worst_delta_pct,within_0_01_pct,worst_gmv_delta_pct
0,54,0.003728,54,0.000023


> **Conclusion.** **Pass.** 54 of 54 weeks, worst delta 0.00004% - a rounding difference of one or two users on a base of tens of thousands. The shares in Part 4 stand.


---

## Part 6 - Check: does reported CAC match its own formula?

**Why it matters.** The sheet states `CAC = Budget RFT_raw / RFT` in the label of
the first row of its block. A stated formula is a claim, and a claim can be
tested: recompute it from the components and compare.

If they disagree, either the formula is not what the column does, or one of the
components is not what its name says.

| | |
|---|---|
| **Built by** | `src/build_marts.py` - `build_qa()` |
| **Written to** | `qa.metric_definition_check` |


In [11]:
q("""
    SELECT channel, week_start,
           ROUND(cac_reported_index, 2)   AS reported_idx,
           ROUND(cac_recomputed_index, 2) AS recomputed_idx,
           ROUND(cac_recomputed_index - cac_reported_index, 3) AS delta_idx
    FROM mart_channel_week_pnl
    WHERE channel_role = 'paid channel'
      AND cac_reported_index IS NOT NULL
      AND cac_reported_index > 0
    ORDER BY ABS(cac_recomputed_index - cac_reported_index) DESC
    LIMIT 12
""")

,channel,week_start,reported_idx,recomputed_idx,delta_idx
0,Tiktok,2025-03-24,24829.67,24921.81,92.139
1,Tiktok,2025-03-17,22462.70,22450.35,-12.357
2,Facebook Ads,2024-12-23,1390.33,1381.21,-9.111
3,Tiktok,2025-10-20,5525.05,5531.54,6.488
4,Tiktok,2025-03-31,5230.78,5234.70,3.921
5,Tiktok,2025-03-10,13144.11,13140.48,-3.636
6,Tiktok,2025-10-27,650.38,651.73,1.348
7,Google Ads,2025-10-27,1153.63,1152.42,-1.205
8,Apple Search Ads,2025-09-22,707.96,706.85,-1.105
9,Facebook Ads,2025-10-06,1187.75,1188.85,1.103


> **Conclusion.** **Pass.** 311 of 317 channel-weeks agree to within one cent on the real figures. The formula is what the column does. That matters for Part 12, because it means the assumption buried in the metric - that 100% of first trips were caused by the spend - is genuinely the assumption in force.


---

## Part 7 - Check: do the two cities sum to "All Viet Nam"?

**Why it matters.** The brand table carries three city scopes: Ha Noi, Ho Chi Minh,
and `All Viet Nam`. The natural reading is that the third is a national figure in
which the two named cities are the largest part.

That reading is testable, and testing it is where this project found something.

| | |
|---|---|
| **Built by** | `src/build_marts.py` - `build_qa()` |
| **Written to** | `qa.city_reconciliation` |


In [12]:
q("""
    SELECT month_start,
           ROUND(national_gmv_index, 3)         AS all_vietnam_idx,
           ROUND(city_gmv_index, 3)             AS hanoi_plus_hcmc_idx,
           ROUND(city_share_of_national * 100, 4) AS cities_as_pct_of_national
    FROM qa_city_reconciliation ORDER BY month_start
""")

,month_start,all_vietnam_idx,hanoi_plus_hcmc_idx,cities_as_pct_of_national
0,2025-01-01,100.000,100.000,100.0
1,2025-02-01,95.144,95.144,100.0
2,2025-03-01,106.972,106.972,100.0
3,2025-04-01,99.295,99.295,100.0
4,2025-05-01,99.769,99.769,100.0


### 100.0000%. Every month. Exactly.

Not 97%, not 99.4% - **exactly 100%**, in all five months, on GMV and on
commission. There is no residual, which means there is no third city.

`All Viet Nam` is not a national total dominated by two cities. It is the sum of
exactly those two and nothing else.


> **Conclusion.** **The label overstates the coverage.** Any share, growth rate or ranking taken from `All Viet Nam` describes two cities. Presented as national - which the label invites - it would be wrong in every slide it reached. A check that passes tells you a number is safe; this one failed usefully, and it only surfaced because the obvious assumption was tested rather than accepted.


---

## Part 8 - Brand economics, and the take rate the source cannot show

**The question.** What does the platform actually keep per brand?

The source gives commission and discount, with discount split by who funded it:
the partner, the merchant, or BE itself. It never subtracts BE's own share from
the commission, because the two sit in different columns.

`net_take_rate = (commission - BE-funded discount) / GMV` is derived here, and it
changes the answer.

| | |
|---|---|
| **Built by** | `src/build_marts.py` - `build_brand_economics()` |
| **Written to** | `mart.brand_month_economics` - 679 rows |


### Who funds the discount


In [13]:
q("""
    SELECT month_start,
           ROUND(SUM(discount_funded_by_be_index))       AS funded_by_be_idx,
           ROUND(SUM(discount_funded_by_merchant_index)) AS funded_by_merchant_idx,
           ROUND(SUM(discount_funded_by_partner_index))  AS funded_by_partner_idx,
           ROUND(100.0 * SUM(discount_funded_by_be_index)
               / NULLIF(SUM(discount_total_index), 0), 1) AS be_share_of_discount_pct
    FROM mart_brand_month_economics WHERE city = 'All Việt Nam'
    GROUP BY month_start ORDER BY month_start
""")

,month_start,funded_by_be_idx,funded_by_merchant_idx,funded_by_partner_idx,be_share_of_discount_pct
0,2025-01-01,6650.0,6650.0,6650.0,100.0
1,2025-02-01,5925.0,6534.0,6206.0,95.8
2,2025-03-01,6520.0,7472.0,5821.0,94.3
3,2025-04-01,5788.0,6462.0,4922.0,95.5
4,2025-05-01,5875.0,6256.0,5240.0,97.5


### Take rate, against take rate net of BE's own subsidy


In [14]:
q("""
    SELECT brand_name,
           ROUND(gmv_index, 1)            AS gmv_idx,
           ROUND(take_rate_index, 2)      AS take_rate_idx,
           ROUND(be_subsidy_rate_index, 2) AS be_subsidy_idx,
           ROUND(net_take_rate_index, 2)  AS net_take_idx
    FROM mart_brand_month_economics
    WHERE city = 'All Việt Nam'
      AND month_start = (SELECT MAX(month_start) FROM mart_brand_month_economics)
    ORDER BY net_take_rate_index LIMIT 12
""")

,brand_name,gmv_idx,take_rate_idx,be_subsidy_idx,net_take_idx
0,Vinamilk,0.4,2.71,9.35,-6.64
1,Café Amazon,2.9,8.06,10.82,-2.77
2,Highlands Coffee,213.7,8.09,10.57,-2.49
3,Gà Rán Popeyes,189.3,8.62,9.91,-1.29
4,The Coffee House,44.2,9.30,10.33,-1.02
5,Burger King,12.7,8.60,8.99,-0.39
6,Phúc Long,279.7,8.00,8.15,-0.15
7,KOI Thé,37.8,7.31,7.25,0.06
8,BUSAN - Món Hàn Quốc,70.3,10.74,10.66,0.09
9,Ô Long Đây,1.5,11.45,11.36,0.09


In [15]:
q("""
    SELECT COUNT(*)                                            AS brands,
           COUNT(*) FILTER (WHERE net_take_rate_index < 0)     AS negative_net_take,
           ROUND(100.0 * SUM(CASE WHEN net_take_rate_index < 0
                             THEN gmv_index ELSE 0 END)
               / SUM(gmv_index), 1)                            AS pct_of_gmv_negative
    FROM mart_brand_month_economics
    WHERE city = 'All Việt Nam'
      AND month_start = (SELECT MAX(month_start) FROM mart_brand_month_economics)
""")

,brands,negative_net_take,pct_of_gmv_negative
0,55,7,11.2


> **Conclusion.** **Seven of 55 brands have a negative net take rate, carrying 11.2% of GMV.** Every one of them looks healthy on the commission line. They are loss-making once BE's own discount funding is netted off, and the delivered table has no column that would reveal it.


---

## Part 9 - The promotion case: what the commission increase bought

**The brief.** The commission rate on the airport route was raised, *"gaining more
revenue and leaving us more rooms to play around with promotional strategies"*.
Design three promotion schemes that grow the business without hurting revenue
per trip.

Before designing anything, the first question is how much room there actually is -
and the twelve days of data answer it.

| | |
|---|---|
| **Built by** | `src/build_marts.py` - `build_promo_economics()` |
| **Written to** | `mart.promo_unit_economics` - 12 days |


In [16]:
q("""
    SELECT day, period,
           ROUND(commission_rate_index, 2)           AS commission_idx,
           ROUND(promo_rate_index, 2)                AS promo_idx,
           ROUND(contribution_margin_rate_index, 2)  AS margin_idx,
           ROUND(gross_spread_rate_index, 2)         AS spread_idx,
           ROUND(contribution_margin_total_index, 1) AS margin_total_idx,
           ROUND(trips_index, 1)                     AS trips_idx
    FROM mart_promo_unit_economics ORDER BY day
""")

,day,period,commission_idx,promo_idx,margin_idx,spread_idx,margin_total_idx,trips_idx
0,2024-01-01,before commission increase,3.11,4.09,-1.01,-0.97,100.0,100.0
1,2024-01-02,before commission increase,3.11,3.76,-0.62,-0.65,69.7,114.3
2,2024-01-03,before commission increase,3.11,4.22,-1.17,-1.10,111.7,98.6
3,2024-01-04,before commission increase,2.40,4.87,-2.80,-2.47,275.1,100.3
4,2024-01-05,before commission increase,2.92,5.64,-3.11,-2.72,322.5,105.8
5,2024-01-06,before commission increase,3.18,5.25,-2.18,-2.08,254.8,119.4
6,2024-01-07,before commission increase,3.37,7.59,-4.83,-4.22,702.8,148.7
7,2024-01-08,before commission increase,3.24,6.23,-3.35,-2.98,457.8,141.0
8,2024-01-09,after commission increase,10.96,7.07,4.83,3.89,-563.3,120.3
9,2024-01-10,after commission increase,10.96,8.37,3.35,2.60,-345.1,105.3


### Read the margin column

It is **negative on all eight days before** the change and **positive on all four
after**. The indexing multiplies by a constant, so the sign is preserved exactly -
that is why a constant was used for ratios rather than a base-100 index, which
would have destroyed it.


In [17]:
q("""
    SELECT period,
           COUNT(*)                                        AS days,
           COUNT(*) FILTER (WHERE contribution_margin_rate_index < 0)
                                                           AS loss_making_days,
           ROUND(AVG(commission_rate_index), 2)            AS avg_commission_idx,
           ROUND(AVG(promo_rate_index), 2)                 AS avg_promo_idx,
           ROUND(AVG(contribution_margin_rate_index), 2)   AS avg_margin_idx
    FROM mart_promo_unit_economics
    GROUP BY period ORDER BY period DESC
""")

,period,days,loss_making_days,avg_commission_idx,avg_promo_idx,avg_margin_idx
0,before commission increase,8,8,3.06,5.21,-2.38
1,after commission increase,4,0,10.99,8.56,3.07


### The part that matters: the pace, not the level

The increase fixed a route that had been losing money on every single observed
day. But look at what happens across the four days after it.


In [18]:
q("""
    SELECT
      ROUND((MAX(contribution_margin_rate_index)
           - MIN(contribution_margin_rate_index)) / 3, 3) AS margin_fall_per_day_idx,
      ROUND((MAX(promo_rate_index)
           - MIN(promo_rate_index)) / 3, 3)               AS promo_rise_per_day_idx,
      ROUND(MIN(contribution_margin_rate_index), 3)       AS margin_on_last_day_idx,
      ROUND(MIN(contribution_margin_rate_index)
          / NULLIF((MAX(contribution_margin_rate_index)
                  - MIN(contribution_margin_rate_index)) / 3, 0), 1)
                                                          AS days_until_zero
    FROM mart_promo_unit_economics
    WHERE period = 'after commission increase'
""")

,margin_fall_per_day_idx,promo_rise_per_day_idx,margin_on_last_day_idx,days_until_zero
0,1.09,0.908,1.557,1.4


> **Conclusion.** On the real figures: margin fell **1.40 points per day** while promotion rose **1.17**. Roughly **3.4 of the 5 points of headroom** the increase created were spent within four days, and on that trajectory the route is loss-making again inside two more. So the answer to "design three promotion schemes" starts with a constraint, not a scheme: the room is about five points and it is already two-thirds committed. Any scheme has to be costed against what is left.


**Four points is not a trend.** The decline is real in the observed window;
whether it continued, stabilised or reversed is unknown. That is stated in
`docs/assumptions.md` rather than left for a reader to work out.


---

## Part 10 - Payback, and every assumption behind it

**The question the brief asked.** Build a model for ROAS and payback time by channel.

No single file can answer it. Acquisition cost is in one workbook, retention in
another, margin in a third - covering three different periods. So this is a
**model**, and the honest version writes its assumptions into the output rather
than hiding them in the code.

| | |
|---|---|
| **Built by** | `src/build_marts.py` - `build_payback()` |
| **Written to** | `mart.channel_payback` - one row per channel |


In [19]:
show_py("build_marts.py", "ASSUMPTIONS")

# build_marts.py, line 342

ASSUMPTIONS = {
    "average_fare": 11.0,            # mart.promo_unit_economics, stable 10.8-11.1
    "trips_per_month": 4.0,          # placeholder - see docs/assumptions.md
    "contribution_margin_rate": 0.04,  # post-increase beAirport CM, conservative
}


In [20]:
q("""
    SELECT channel,
           ROUND(cac_index, 1)               AS cac_idx,
           ROUND(first_trip_users_index, 1)  AS rft_idx,
           ROUND(assumed_m1_retention_index, 1) AS m1_retention_idx,
           assumed_trips_per_month,
           assumed_margin_rate,
           ROUND(payback_months_no_churn, 2)   AS payback_months_simple,
           ROUND(payback_months_with_churn, 2) AS payback_months_with_churn
    FROM mart_channel_payback ORDER BY payback_months_with_churn
""")

,channel,cac_idx,rft_idx,m1_retention_idx,assumed_trips_per_month,assumed_margin_rate,payback_months_simple,payback_months_with_churn
0,Others,0.0,4.4,61.4,4.0,0.04,0.00,0.00
1,Affiliate,337.6,399.8,61.4,4.0,0.04,0.90,2.63
2,Tiktok,806.8,5.1,61.4,4.0,0.04,2.15,6.29
3,Apple Search Ads,860.0,68.5,61.4,4.0,0.04,2.29,6.71
4,Google Ads,984.2,55.5,61.4,4.0,0.04,2.62,7.68
5,Facebook Ads,1058.1,66.8,61.4,4.0,0.04,2.82,8.25


### Two things in that table are not results

**`Others` shows a payback of zero.** It has 15,393 first-trip users and **no
attributed budget at all**, so its CAC computes to zero. That is an attribution
gap, not a performance result. It is left visible and labelled rather than
quietly dropped, because a reader who finds it themselves will wonder what else
was removed.

**`assumed_trips_per_month = 4.0` is a placeholder.** Nothing in the data supports
it. It is the one input that is neither measured nor bounded by something
observed - and the only one that can change the *ranking*, because a channel
acquiring daily commuters and a channel acquiring occasional airport users do
not pay back at the same speed even at identical CAC.

Average fare and margin rate also scale every payback period, but they scale all
of them equally, so the ordering survives. Trip frequency does not.


> **Conclusion.** The ranking - Affiliate fastest, Facebook Ads slowest - is really a CAC ranking with a constant applied to it. Saying so is the difference between a model and a result. `docs/assumptions.md` states each input, where it came from, and what happens to the conclusion if it is wrong.


---

## Part 11 - The indexing layer

**The constraint.** The source is BE Group's own operating data. No real figure
should be published, but a portfolio project that blanks its numbers out cannot
be read.

So every published table is **transformed, not redacted** - and the transform is
chosen so every conclusion above survives it exactly.

| | |
|---|---|
| **Built by** | `src/publish_indexed.py` |
| **Written to** | `data/published/*.parquet` and `*.csv` |


| Column type | Rule | What survives |
|---|---|---|
| **Levels** - GMV, cost, users, orders, commission | rescaled so the first period of the series = 100 | growth rates, shares, trends |
| **Ratios** - CAC, CIR, take rate, margin, conversion, retention | multiplied by one undisclosed constant per metric family | every within-family comparison, **including the sign** |
| **Reconciliation deltas** expressed as a share | published as measured | a delta of 0.00% is the evidence a check passed and discloses nothing |

### Why a constant for ratios rather than an index

Several of these series cross zero - contribution margin most importantly.
Rebasing a series that crosses zero destroys the sign, and in Part 9 **the sign
is the finding**. A multiplicative constant preserves sign, ratio and crossing
point, and hides only the level.


In [21]:
show_py("publish_indexed.py", "transform")

# publish_indexed.py, line 169

def transform(frame: pd.DataFrame, factors: dict[str, float]) -> pd.DataFrame:
    period_column = next(
        (c for c in ("week_start", "month_start", "day") if c in frame.columns), None
    )
    bases: dict[str, float] = {}
    for group in SHARED_BASE_GROUPS:
        if group & set(frame.columns):
            base = shared_base(frame, group, period_column)
            bases.update({column: base for column in group})

    out = pd.DataFrame(index=frame.index)
    for column in frame.columns:
        if column in PASS_THROUGH:
            out[column] = frame[column]
        elif column in bases:
            out[f"{column}_index"] = (
                pd.to_numeric(frame[column], errors="coerce") / bases[column] * 100
            ).round(3)
        elif column in LEVEL_COLUMNS:
            out[f"{column}_index"] = index_levels(frame, column, period_column)
        elif (family := family_of(column)) is not None:
            out[f"{column}_index"] = (
 

### Proof that the findings survive it


In [22]:
checks = {
    "margin still crosses zero on 2024-01-09":
        q("""SELECT COUNT(*) AS n FROM mart_promo_unit_economics
             WHERE (period = 'before commission increase'
                    AND contribution_margin_rate_index < 0)
                OR (period = 'after commission increase'
                    AND contribution_margin_rate_index > 0)""").iat[0, 0] == 12,
    "channel payback ranking unchanged":
        q("""SELECT channel FROM mart_channel_payback
             WHERE channel != 'Others'
             ORDER BY payback_months_with_churn""")["channel"].tolist()
        == ["Affiliate", "Tiktok", "Apple Search Ads", "Google Ads", "Facebook Ads"],
    "city reconciliation still exactly 100%":
        bool((q("SELECT city_share_of_national AS s FROM qa_city_reconciliation")["s"]
               .round(6) == 1.0).all()),
    "seven brands still negative on net take":
        q("""SELECT COUNT(*) AS n FROM mart_brand_month_economics
             WHERE city = 'All Việt Nam' AND net_take_rate_index < 0
               AND month_start = (SELECT MAX(month_start)
                                  FROM mart_brand_month_economics)""").iat[0, 0] == 7,
}
for claim, holds in checks.items():
    print(f"  {'PASS' if holds else 'FAIL'}  {claim}")

  PASS  margin still crosses zero on 2024-01-09
  PASS  channel payback ranking unchanged
  PASS  city reconciliation still exactly 100%
  PASS  seven brands still negative on net take


> **Conclusion.** Every finding in this notebook is reproducible from the published, indexed files alone. The real values never left `data/private/`, which is git-ignored along with the source workbooks.


---

## Part 12 - What this data cannot answer

**The brief's third question:** how would you estimate the incremental impact of
increasing spend on one channel, from this data alone?

Largely, you cannot - and saying so precisely is the deliverable.

| | |
|---|---|
| **Built by** | `docs/measurement_design.md` |
| **Written to** | a design, not a number |


### The assumption nobody states

```
CAC = Budget RFT_raw / RFT
```

That formula divides the budget by **every** first trip the channel recorded. It
therefore assumes 100% of them were caused by the spend. Part 6 confirmed the
column really does compute this, so the assumption is genuinely in force - it is
just invisible, because it lives inside the metric's definition rather than
beside it.

### Why Affiliate in particular

Two thirds of paid acquisition at a third of the cost per user of anything else.
Two explanations fit, and nothing here separates them:

1. Affiliate is genuinely the most efficient channel BE has.
2. Affiliate is capturing users who were going to install anyway, through a
   trackable link that claims the credit.

### Why a regression will not settle it

| | |
|---|---|
| **Confounding** | budgets rise in high-demand weeks, so heavy-spend weeks also carry more organic demand. The model attributes the season to the budget |
| **Reverse causality** | the budget is *set in response to* performance. Good performance causes high spend as well as the other way round. In this dataset the budget rows are weekly decisions made by a team looking at exactly these numbers |

A regression run anyway returns a confident coefficient, a tight interval and a
plausible chart. The number is meaningless. **Producing it is the mistake;
declining to, and specifying the test that would work, is the job.**


### The design, in brief

| | |
|---|---|
| **Unit of randomisation** | geography - district or service zone, **not the user** |
| **Treatment** | Affiliate spend switched off or cut by a fixed share |
| **Primary metric** | first-trip users **across all channels**, not Affiliate's own number |
| **Guardrails** | completion rate, cancellation rate, M1 retention of the test cohort |
| **The constraint** | this extract has two city scopes. Two geographies cannot support a geo holdout - it needs districts, or the switchback design instead |

That last row is the point. **Establish the sample constraint before proposing
the test, not during it.** Full specification, including the minimum detectable
effect implied by the 0.221 coefficient of variation in weekly first trips, is in
`docs/measurement_design.md`.


> **Conclusion.** The budget-cut answer follows from this. Cut from Facebook and Google, which carry 38.8% of budget for 20.4% of first trips - but do **not** move the saving into Affiliate. Hold it flat and use the cut as a natural experiment: if total acquisition falls by less than the cut channels' attributed share, that is the first evidence anyone has had about incrementality here, and it costs nothing extra to collect.


---

## What this project demonstrates

| | |
|---|---|
| **Working with a real handover** | merged-cell, metric-as-row Excel parsed reproducibly, with four traps handled explicitly and written down |
| **Checking before quoting** | four checks, of which one found that `All Viet Nam` is two cities |
| **Commercial reasoning** | net take rate, discount funding by party, contribution margin per trip, payback |
| **Scope discipline** | a model labelled as a model, a placeholder named as a placeholder, an attribution gap left visible |
| **Measurement judgement** | declining to produce a confounded incrementality estimate, and specifying the test that would work |
| **Handling confidential data** | a transform chosen so every conclusion survives and no level is recoverable, verified rather than assumed |

### Limitations, stated

- The three source files cover **three different periods**. The payback model
  combines them anyway and says so.
- `assumed_trips_per_month` is a placeholder and would change the ranking.
- The promotion case is **twelve days**, eight before and four after, with no
  control route. The margin decline is a description, not a causal estimate.
- `dd` could not be identified and is excluded from every calculation.
- `All Viet Nam` covers two cities. Every share derived from it inherits that.

---

*Dictionary: `docs/data_dictionary.md` - Assumptions: `docs/assumptions.md`*  
*Measurement design: `docs/measurement_design.md` - Original submission: `docs/original_submission.md`*


In [23]:
db.close()
print("Walkthrough complete.")

Walkthrough complete.
